# Silver Transform Job
Reads cleaned-but-raw event data from the **bronze** parquet layer, applies the agreed cleaning rules, and loads the result into the **silver** schema in Postgres. Rows that fail validation are routed to a **DLQ** table instead of being dropped or crashing the job. Processing is **incremental**: a manifest table in Postgres tracks which bronze files have already been loaded, so re-running this job never reprocesses the same file twice, and each file is capped at a batch of 20 per run so a large backlog drains gradually instead of all at once.

## 1. Imports

In [56]:
import duckdb
import os
from dotenv import load_dotenv

# override=True ensures freshly-edited .env values are picked up even if
# this kernel already loaded an older version of the environment.
load_dotenv(override=True)

True

## 2. Connections
### 2.1 Postgres credentials — read from environment variables (`.env`), never hardcoded.

In [57]:
pg_host = os.environ["POSTGRES_HOST"]
pg_port = os.environ["POSTGRES_PORT"]
pg_db = os.environ["POSTGRES_DB"]
pg_user = os.environ["POSTGRES_USER"]
pg_password = os.environ["POSTGRES_PASSWORD"]

### 2.2 DuckDB connection + Postgres attachment
DuckDB is the processing engine only — no persistent storage here. `ATTACH ... TYPE postgres` opens a live connection aliased as `pg`; any SQL prefixed `pg.` runs directly against the real Postgres database.

In [58]:
con = duckdb.connect()
con.execute("INSTALL postgres; LOAD postgres;")
con.execute(f"""
    ATTACH 'dbname={pg_db} user={pg_user} host={pg_host} password={pg_password} port={pg_port}'
    AS pg (TYPE postgres);
""")

## 3. Schema Creation
- **`silver`** — cleaned event table + DLQ table.
- **`meta`** — the manifest table. Kept separate from `silver` since it's pipeline bookkeeping, not business data.

In [59]:
con.execute("CREATE SCHEMA IF NOT EXISTS pg.silver;")

In [60]:
con.execute("CREATE SCHEMA IF NOT EXISTS pg.meta;")

## 4. Table Creation
### 4.1 Silver table
No `PRIMARY KEY` — uniqueness is enforced logically by the dedup step (composite match on `user_id + product_id + event_type + event_time`), not structurally by the table.

In [61]:
con.execute("""
    CREATE TABLE IF NOT EXISTS pg.silver.silver_table (
        event_time      TIMESTAMP,
        event_type      TEXT,
        product_id      TEXT,
        category_id     TEXT,
        category_code   TEXT,
        brand           TEXT,
        price           NUMERIC,
        user_id         TEXT,
        user_session    TEXT,
        source_file     TEXT,
        transformed_at  TIMESTAMP DEFAULT now()
    );
""")

### 4.2 Manifest table
Lives in Postgres (not DuckDB) so it can be updated in the *same transaction* as the silver/DLQ inserts — guaranteeing the data and its tracking state can never disagree, even if the job crashes mid-file.

In [62]:
con.execute("""
    CREATE TABLE IF NOT EXISTS pg.meta.silver_processed_files (
        file_name        TEXT PRIMARY KEY,
        status            TEXT NOT NULL,
        row_count_total   INTEGER,
        row_count_silver  INTEGER,
        row_count_dlq     INTEGER,
        processed_at      TIMESTAMP DEFAULT now()
    );
""")

### 4.3 DLQ table
Rejected rows are never dropped and never crash the job — original values are preserved as JSON (schema-agnostic, so it absorbs any row shape), with a reason code explaining the rejection.

In [63]:
con.execute("""
    CREATE TABLE IF NOT EXISTS pg.silver.dlq_events (
        raw_data          JSON,
        rejection_reason  TEXT,
        source_file       TEXT,
        quarantined_at    TIMESTAMP DEFAULT now(),
        reprocessed       BOOLEAN DEFAULT false
    );
""")

## 5. Source Path
Bronze parquet files sit in a Hive-partitioned folder tree written by `bronze-ingestion`. Path is relative to this notebook's own location.

In [64]:
bronze_path = "../bronze-jobs/data/raw-data/**/*.parquet"
BATCH_SIZE = 20  # our agreed cap on files processed per run

## 6. Get Pending Files
Compares bronze files on disk against the manifest's `success` entries. Anything not yet marked `success` is pending, capped at `BATCH_SIZE` so a single run stays bounded even with a large backlog.

In [65]:
pending_files = con.execute(f"""
    SELECT DISTINCT filename
    FROM read_parquet('{bronze_path}', filename=True)
    WHERE filename NOT IN (
        SELECT file_name FROM pg.meta.silver_processed_files WHERE status = 'success'
    )
    LIMIT {BATCH_SIZE}
""").fetchall()

pending_files = [row[0] for row in pending_files]
print(f"{len(pending_files)} file(s) pending this run:")
print(pending_files)

6 file(s) pending this run:
['..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00005.parquet', '..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00003.parquet', '..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00000.parquet', '..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00002.parquet', '..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00001.parquet', '..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00004.parquet']


## 7. Process the Batch — Loop Over All Pending Files
This is the production version of the job: each file goes through the full cleaning sequence (select columns → flag missing IDs → trim → lowercase → normalize *[not applicable here]* → cast types → dedup → split) and is then loaded, all wrapped in a **single transaction per file**.

**Why one transaction per file:** if anything fails partway through a file (a bad insert, a connection hiccup), `ROLLBACK` undoes every change made for that file — the silver insert, the DLQ insert, and the manifest update all either land together or not at all. That file simply stays absent from the manifest, so the *next* run will correctly retry it. No duplicates, no half-loaded files, no silent data loss.

**Why process one file, not the whole folder, per query:** keeps memory use bounded and gives us a natural, resumable unit of work — exactly the chunking discussion we had earlier, just at the file level instead of the row level.

In [66]:
run_summary = []  # collects a per-file summary to print at the end

for current_file in pending_files:
    print(f"Processing: {current_file}")

    try:
        # Start a transaction: the silver insert, DLQ insert, and manifest
        # update below either all commit together, or none do.
        con.execute("BEGIN TRANSACTION;")

        # --- Steps 1-6: select, flag missing IDs, trim, lowercase, cast types ---
        # --- Step 7: dedup on (user_id, product_id, event_type, event_time)  ---
        # --- Step 8a: clean rows -> silver_table                              ---
        con.execute(f"""
            INSERT INTO pg.silver.silver_table
            (event_time, event_type, product_id, category_id, category_code, brand, price, user_id, user_session, source_file, transformed_at)
            WITH cleaned AS (
                SELECT
                    try_cast(event_time AS TIMESTAMP)  AS event_time,
                    lower(trim(event_type))     AS event_type,
                    lower(trim(product_id))     AS product_id,
                    lower(trim(category_id))    AS category_id,
                    lower(trim(category_code))  AS category_code,
                    lower(trim(brand))          AS brand,
                    try_cast(price AS DOUBLE)    AS price,
                    lower(trim(user_id))         AS user_id,
                    lower(trim(user_session))    AS user_session,
                    filename AS source_file,
                    now() AS transformed_at,
                    CASE
                        WHEN trim(user_id) IS NULL OR trim(user_id) = '' THEN 'missing_identifier'
                        WHEN try_cast(event_time AS TIMESTAMP) IS NULL THEN 'type_cast_failure'
                        WHEN try_cast(price AS DOUBLE) IS NULL THEN 'type_cast_failure'
                        ELSE NULL
                    END AS rejection_reason
                FROM read_parquet('{current_file}', filename=True)
            ),
            deduped AS (
                SELECT *,
                    ROW_NUMBER() OVER (
                        PARTITION BY user_id, product_id, event_type, event_time
                        ORDER BY transformed_at DESC
                    ) AS rn
                FROM cleaned
            )
            SELECT event_time, event_type, product_id, category_id, category_code, brand, price, user_id, user_session, source_file, transformed_at
            FROM deduped
            WHERE rn = 1 AND rejection_reason IS NULL
        """)

        # --- Step 8b: bad rows -> dlq_events, original values kept as JSON ---
        con.execute(f"""
            INSERT INTO pg.silver.dlq_events
            (raw_data, rejection_reason, source_file, quarantined_at)
            WITH cleaned AS (
                SELECT
                    event_time, event_type, product_id, category_id, category_code, brand, price, user_id, user_session,
                    filename AS source_file,
                    now() AS transformed_at,
                    CASE
                        WHEN trim(user_id) IS NULL OR trim(user_id) = '' THEN 'missing_identifier'
                        WHEN try_cast(event_time AS TIMESTAMP) IS NULL THEN 'type_cast_failure'
                        WHEN try_cast(price AS DOUBLE) IS NULL THEN 'type_cast_failure'
                        ELSE NULL
                    END AS rejection_reason
                FROM read_parquet('{current_file}', filename=True)
            ),
            deduped AS (
                SELECT *,
                    ROW_NUMBER() OVER (
                        PARTITION BY user_id, product_id, event_type, event_time
                        ORDER BY transformed_at DESC
                    ) AS rn
                FROM cleaned
            )
            SELECT
                to_json(struct_pack(event_time, event_type, product_id, category_id, category_code, brand, price, user_id, user_session)) AS raw_data,
                rejection_reason,
                source_file,
                now() AS quarantined_at
            FROM deduped
            WHERE rn = 1 AND rejection_reason IS NOT NULL
        """)

        # --- Row counts for this file, used for the manifest entry ---
        total_count = con.execute(
            f"SELECT count(*) FROM read_parquet('{current_file}', filename=True)"
        ).fetchone()[0]
        silver_count = con.execute(
            "SELECT count(*) FROM pg.silver.silver_table WHERE source_file = ?;", [current_file]
        ).fetchone()[0]
        dlq_count = con.execute(
            "SELECT count(*) FROM pg.silver.dlq_events WHERE source_file = ?;", [current_file]
        ).fetchone()[0]

        # --- Manifest update: only reached if both inserts above succeeded ---
        con.execute("""
            INSERT INTO pg.meta.silver_processed_files
            (file_name, status, row_count_total, row_count_silver, row_count_dlq, processed_at)
            VALUES (?, 'success', ?, ?, ?, now())
            ON CONFLICT (file_name) DO UPDATE SET
                status = 'success',
                row_count_total = excluded.row_count_total,
                row_count_silver = excluded.row_count_silver,
                row_count_dlq = excluded.row_count_dlq,
                processed_at = excluded.processed_at;
        """, [current_file, total_count, silver_count, dlq_count])

        con.execute("COMMIT;")

        run_summary.append({
            "file": current_file,
            "status": "success",
            "total": total_count,
            "silver": silver_count,
            "dlq": dlq_count,
        })
        print(f"  -> success | total={total_count} silver={silver_count} dlq={dlq_count}")

    except Exception as e:
        # Undo every change made for this file - silver, DLQ, and manifest
        # inserts are all rolled back together, so nothing is half-applied.
        con.execute("ROLLBACK;")
        run_summary.append({"file": current_file, "status": "failed", "error": str(e)})
        print(f"  -> FAILED: {e}")
        # Note: this file is NOT marked in the manifest, so the next run
        # will automatically retry it.

Processing: ..\bronze-jobs\data\raw-data\source=2019-Nov\ingest_date=2026-09-08\chunk_00005.parquet
  -> success | total=48575 silver=48548 dlq=0
Processing: ..\bronze-jobs\data\raw-data\source=2019-Nov\ingest_date=2026-09-08\chunk_00003.parquet
  -> success | total=200000 silver=399732 dlq=0
Processing: ..\bronze-jobs\data\raw-data\source=2019-Nov\ingest_date=2026-09-08\chunk_00000.parquet
  -> success | total=200000 silver=199894 dlq=0
Processing: ..\bronze-jobs\data\raw-data\source=2019-Nov\ingest_date=2026-09-08\chunk_00002.parquet
  -> success | total=200000 silver=199866 dlq=0
Processing: ..\bronze-jobs\data\raw-data\source=2019-Nov\ingest_date=2026-09-08\chunk_00001.parquet
  -> success | total=200000 silver=199882 dlq=0
Processing: ..\bronze-jobs\data\raw-data\source=2019-Nov\ingest_date=2026-09-08\chunk_00004.parquet
  -> success | total=200000 silver=199907 dlq=0


## 8. Run Summary
Quick printout of what happened this run: files processed, rows loaded to silver, rows quarantined to DLQ, and any failures.

In [67]:
print("\n=== Run Summary ===")
for entry in run_summary:
    print(entry)

succeeded = sum(1 for e in run_summary if e["status"] == "success")
failed = sum(1 for e in run_summary if e["status"] == "failed")
total_silver = sum(e.get("silver", 0) for e in run_summary if e["status"] == "success")
total_dlq = sum(e.get("dlq", 0) for e in run_summary if e["status"] == "success")

print(f"\nFiles succeeded: {succeeded} | Files failed: {failed}")
print(f"Total rows -> silver: {total_silver} | Total rows -> DLQ: {total_dlq}")


=== Run Summary ===
{'file': '..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00005.parquet', 'status': 'success', 'total': 48575, 'silver': 48548, 'dlq': 0}
{'file': '..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00003.parquet', 'status': 'success', 'total': 200000, 'silver': 399732, 'dlq': 0}
{'file': '..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00000.parquet', 'status': 'success', 'total': 200000, 'silver': 199894, 'dlq': 0}
{'file': '..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00002.parquet', 'status': 'success', 'total': 200000, 'silver': 199866, 'dlq': 0}
{'file': '..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00001.parquet', 'status': 'success', 'total': 200000, 'silver': 199882, 'dlq': 0}
{'file': '..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00004.parquet', 'status': 'succ

## 9. Verify
Sanity checks after the run: manifest contents, and overall table counts.

In [68]:
con.execute("SELECT * FROM pg.meta.silver_processed_files ORDER BY processed_at;").fetchall()

[('..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00005.parquet',
  'success',
  48575,
  48548,
  0,
  datetime.datetime(2026, 9, 9, 16, 40, 47, 228305)),
 ('..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00003.parquet',
  'success',
  200000,
  399732,
  0,
  datetime.datetime(2026, 9, 9, 16, 40, 47, 590718)),
 ('..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00000.parquet',
  'success',
  200000,
  199894,
  0,
  datetime.datetime(2026, 9, 9, 16, 40, 48, 664793)),
 ('..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00002.parquet',
  'success',
  200000,
  199866,
  0,
  datetime.datetime(2026, 9, 9, 16, 40, 49, 770693)),
 ('..\\bronze-jobs\\data\\raw-data\\source=2019-Nov\\ingest_date=2026-09-08\\chunk_00001.parquet',
  'success',
  200000,
  199882,
  0,
  datetime.datetime(2026, 9, 9, 16, 40, 51, 261412)),
 ('..\\bronze-jobs\\data\\raw-data\\source=2019

In [69]:
con.execute("SELECT count(*) FROM pg.silver.silver_table;").fetchall()

[(1247829,)]

In [70]:
con.execute("SELECT count(*) FROM pg.silver.dlq_events;").fetchall()

[(0,)]

## 10. Resumability Test (optional, manual)
To confirm the incremental design actually works: interrupt the kernel partway through a run with a larger backlog (e.g. kill it after 2-3 files have committed), then re-run cells 6 and 7. The pending-files query should return only the files that never made it into the manifest — already-loaded files are correctly skipped, and no duplicate rows appear in `silver_table`.

## 11. Next Steps
This notebook is the validated, transactional, batch-processing version of the job — the last step before deployment:

1. Refactor into a `.py` script with real functions (`get_pending_files()`, `process_file(con, file)`, `run()`), so it can be scheduled (cron, Airflow, etc.) instead of run manually from a notebook.
2. Add proper logging (instead of `print`) for production monitoring.
3. Consider alerting if `row_count_dlq` for a file crosses some threshold — a spike in rejections is often a sign of an upstream schema change.